In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    BaggingClassifier, AdaBoostClassifier,
    VotingClassifier, StackingClassifier, ExtraTreesClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
import time


In [3]:
df = pd.read_csv("C:\\Users\\nakul\\Downloads\\taiwanese+bankruptcy+prediction\\data.csv")

FEATURE_MAP = {
    'Cash flow rate':           ' Cash flow rate',
    'Cash Flow to Sales':       ' Cash Flow to Sales',
    'Cash Flow to Liability':   ' Cash Flow to Liability',
    'Current Ratio':            ' Current Ratio',
    'Quick Ratio':              ' Quick Ratio',
    'Cash/Current Liability':   ' Cash/Current Liability',
    'Debt ratio %':             ' Debt ratio %',
    'Liability to Equity':      ' Liability to Equity',
    'Interest Coverage Ratio':  ' Interest Coverage Ratio (Interest expense to EBIT)',
    'DFL':                      ' Degree of Financial Leverage (DFL)',
    'ROA':                      ' ROA(C) before interest and depreciation before interest',
    'Operating Gross Margin':   ' Operating Gross Margin',
    'Gross Profit to Sales':    ' Gross Profit to Sales',
    'Net Income to Total Assets':' Net Income to Total Assets',
    'Revenue Growth Rate':      ' Total Asset Growth Rate',
    'Accounts Receivable Turnover': ' Accounts Receivable Turnover',
    'Inventory Turnover Rate':  ' Inventory Turnover Rate (times)',
    'Average Collection Days':  ' Average Collection Days',
}

feature_cols = list(FEATURE_MAP.values())
X = df[feature_cols].copy()
X.columns = list(FEATURE_MAP.keys())
y = df['Bankrupt?'].copy()

print(f"\n  Dataset      : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Features used: {len(feature_cols)}")
print(f"  Class 0 (Safe)    : {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"  Class 1 (Bankrupt): {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")
print(f"  Imbalance ratio  : 1 : {int((y==0).sum()/(y==1).sum())}")


  Dataset      : 6819 rows × 96 columns
  Features used: 18
  Class 0 (Safe)    : 6599 (96.8%)
  Class 1 (Bankrupt): 220 (3.2%)
  Imbalance ratio  : 1 : 29


In [4]:
def manual_smote(X, y, k=5, random_state=42):
    """Synthetic Minority Over-sampling Technique"""
    rng = np.random.RandomState(random_state)
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    X_min = X[y == 1].values
    n_synthetic = (y == 0).sum() - (y == 1).sum()
    synthetic = []
    for _ in range(n_synthetic):
        idx = rng.randint(0, len(X_min))
        sample = X_min[idx]
        # pick random neighbour from k nearest
        dists = np.linalg.norm(X_min - sample, axis=1)
        dists[idx] = np.inf
        neighbours = np.argsort(dists)[:k]
        nn_idx = rng.choice(neighbours)
        gap = rng.random()
        synthetic.append(sample + gap * (X_min[nn_idx] - sample))
    X_syn = np.vstack(synthetic)
    X_res = np.vstack([X.values, X_syn])
    y_res = np.hstack([y.values, np.ones(len(X_syn), dtype=int)])
    return pd.DataFrame(X_res, columns=X.columns), pd.Series(y_res)


X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_raw), columns=X.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X.columns
)

print("\n  Applying SMOTE to training set...")
X_train_sm, y_train_sm = manual_smote(X_train_scaled, y_train_raw, k=5)
print(f"  After SMOTE — Class 0: {(y_train_sm==0).sum()}, Class 1: {(y_train_sm==1).sum()}")


  Applying SMOTE to training set...
  After SMOTE — Class 0: 5279, Class 1: 5279


In [5]:
BASE_MODELS = {
    'Logistic Regression':   LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree':         DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'KNN':                   KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes':           GaussianNB(),
    'SVM':                   SVC(probability=True, class_weight='balanced', kernel='rbf', random_state=42),
}

ENSEMBLE_MODELS = {
    'Random Forest':         RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42),
    'Extra Trees':           ExtraTreesClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    'AdaBoost':              AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'Bagging (DT)':          BaggingClassifier(
                                 estimator=DecisionTreeClassifier(max_depth=6),
                                 n_estimators=100, random_state=42
                             ),
}

# Voting Classifier (Soft)
voting_soft = VotingClassifier(estimators=[
    ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ('gb',  GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('lr',  LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
    ('svm', SVC(probability=True, class_weight='balanced', random_state=42)),
], voting='soft')

# Voting Classifier (Hard)
voting_hard = VotingClassifier(estimators=[
    ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ('gb',  GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('et',  ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ('dt',  DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)),
], voting='hard')

# Stacking (Level-0 diverse, Level-1 meta)
stacking = StackingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
        ('gb',  GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('et',  ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
        ('svm', SVC(probability=True, class_weight='balanced', random_state=42)),
        ('lr',  LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=7)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5, random_state=42),
    cv=5, passthrough=False, n_jobs=-1
)
# Stacking with GBM meta-learner
stacking_gbm = StackingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
        ('gb',  GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('et',  ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
        ('dt',  DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)),
    ],
    final_estimator=GradientBoostingClassifier(n_estimators=50, random_state=42),
    cv=5, passthrough=True, n_jobs=-1
)

ADVANCED_MODELS = {
    'Voting (Soft)':     voting_soft,
    'Voting (Hard)':     voting_hard,
    'Stacking (LR meta)':stacking,
    'Stacking (GBM meta)':stacking_gbm,
}

ALL_MODELS = {**BASE_MODELS, **ENSEMBLE_MODELS, **ADVANCED_MODELS}

In [11]:
# Stacking with GBM meta-learner
stacking_gbm = StackingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
        ('gb',  GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('et',  ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
        ('dt',  DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)),
    ],
    final_estimator=GradientBoostingClassifier(n_estimators=50, random_state=42),
    cv=5, passthrough=True, n_jobs=-1
)

ADVANCED_MODELS = {
    'Voting (Soft)':     voting_soft,
    'Voting (Hard)':     voting_hard,
    'Stacking (LR meta)':stacking,
    'Stacking (GBM meta)':stacking_gbm,
}

ALL_MODELS = {**BASE_MODELS, **ENSEMBLE_MODELS, **ADVANCED_MODELS}
# ─────────────────────────────────────────────
# 5. TRAIN & EVALUATE ALL MODELS
# ─────────────────────────────────────────────
results = []
conf_matrices = {}
roc_data = {}

print(f"\n  Training {len(ALL_MODELS)} models...\n")
print(f"  {'Model':<25} {'Acc':>7} {'F1':>7} {'Prec':>7} {'Rec':>7} {'AUC':>7} {'Time':>7}")
print("  " + "-"*65)

for name, model in ALL_MODELS.items():
    t0 = time.time()
    model.fit(X_train_sm, y_train_sm)
    train_time = time.time() - t0

    y_pred = model.predict(X_test_scaled)
    try:
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    except:
        y_prob = None

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

    results.append({
        'Model': name, 'Accuracy': acc, 'F1 Score': f1,
        'Precision': prec, 'Recall': rec, 'ROC-AUC': auc,
        'Train Time (s)': round(train_time, 2),
        'Category': 'Baseline' if name in BASE_MODELS else
                    ('Ensemble' if name in ENSEMBLE_MODELS else 'Advanced')
    })
    conf_matrices[name] = confusion_matrix(y_test, y_pred)
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_data[name] = (fpr, tpr, auc)

    print(f"  {name:<25} {acc:>7.4f} {f1:>7.4f} {prec:>7.4f} {rec:>7.4f} {auc:>7.4f} {train_time:>6.1f}s")

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False).reset_index(drop=True)
results_df['Rank'] = results_df.index + 1


  Training 14 models...

  Model                         Acc      F1    Prec     Rec     AUC    Time
  -----------------------------------------------------------------
  Logistic Regression        0.4553  0.0559  0.0296  0.5000  0.5153    0.2s
  Decision Tree              0.8966  0.3188  0.2025  0.7500  0.8611    0.1s
  KNN                        0.8783  0.2783  0.1720  0.7273  0.8570    0.0s
  Naive Bayes                0.0374  0.0615  0.0317  0.9773  0.5228    0.0s
  SVM                        0.9648  0.1111  0.3000  0.0682  0.2343   29.9s
  Random Forest              0.9443  0.4571  0.3333  0.7273  0.9298    5.6s
  Extra Trees                0.9641  0.5421  0.4603  0.6591  0.9403    1.6s
  Gradient Boosting          0.9230  0.3636  0.2479  0.6818  0.9199   16.0s
  AdaBoost                   0.8358  0.2483  0.1457  0.8409  0.9200    2.7s
  Bagging (DT)               0.9135  0.3656  0.2394  0.7727  0.9290    7.9s
  Voting (Soft)              0.9318  0.4151  0.2870  0.7500  0.9325   